# PLS Pipeline
`CSV → Correlation Filter → PLS → VIP Score → Top N Selection → Final PLS`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import pearsonr
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict, LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries loaded')

---
## ⚙️ CONFIG

In [ ]:
# ================================================================
FILE_PATH      = 'my_features.csv'  # file CSV: cột đầu = y, còn lại = features
TARGET_COL     = 'glucose_mM'       # tên cột y

# --- Correlation filter ---
CORR_THRESHOLD = 0.85               # giữ features có |r| >= ngưỡng này

# --- PLS ---
MAX_COMPONENTS = 2                  # giới hạn chống overfit

# --- VIP → Top N selection ---
VIP_TOP_N      = 5                  # chọn N features VIP cao nhất
# ================================================================
print(f'File        : {FILE_PATH}')
print(f'Target      : {TARGET_COL}')
print(f'Corr filter : |r| >= {CORR_THRESHOLD}')
print(f'Max comp    : {MAX_COMPONENTS}')
print(f'VIP Top N   : {VIP_TOP_N}')

---
## 1. Load Data

In [ ]:
df = pd.read_csv(FILE_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
display(df)

y            = df[TARGET_COL].values
feature_cols = [c for c in df.columns if c != TARGET_COL]
X_all        = df[feature_cols].values.astype(float)

print(f'\nSamples  : {X_all.shape[0]}')
print(f'Features : {X_all.shape[1]}')
print(f'y        : {y}')

---
## 2. Correlation Filter

In [ ]:
corr_rows = []
kept_idx  = []   # index của feature được giữ lại

for i, col in enumerate(feature_cols):
    vals = X_all[:, i]
    if np.any(np.isnan(vals)) or np.std(vals) == 0:
        continue
    r, p = pearsonr(vals, y)
    corr_rows.append({'feature': col, 'pearson_r': round(r,4), 'abs_r': round(abs(r),4)})
    if abs(r) >= CORR_THRESHOLD:
        kept_idx.append(i)

corr_df    = pd.DataFrame(corr_rows).sort_values('abs_r', ascending=False)
kept_names = [feature_cols[i] for i in kept_idx]
X_filt     = X_all[:, kept_idx]

print(f'Total features  : {len(feature_cols)}')
print(f'After |r|>={CORR_THRESHOLD}: {len(kept_idx)} kept,  {len(feature_cols)-len(kept_idx)} removed')
print(f'\nAll features sorted by |r|:')
print(corr_df.to_string(index=False))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.hist(corr_df['abs_r'], bins=20, color='steelblue', alpha=0.8, edgecolor='white')
ax1.axvline(CORR_THRESHOLD, color='red', lw=2, linestyle='--', label=f'threshold={CORR_THRESHOLD}')
ax1.set_xlabel('|Pearson r|'); ax1.set_ylabel('Count')
ax1.set_title('Correlation Distribution', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

top_plot   = corr_df.head(min(25, len(corr_df)))
bar_colors = ['#e74c3c' if r >= CORR_THRESHOLD else '#95a5a6' for r in top_plot['abs_r']]
ax2.barh(range(len(top_plot)), top_plot['abs_r'], color=bar_colors, alpha=0.85, edgecolor='black')
ax2.set_yticks(range(len(top_plot)))
ax2.set_yticklabels(top_plot['feature'], fontsize=8)
ax2.invert_yaxis()
ax2.axvline(CORR_THRESHOLD, color='red', lw=1.5, linestyle='--')
ax2.set_xlabel('|Pearson r|'); ax2.set_title('Features by |r|  (red = kept)', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('pipeline_corr_filter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. PLS với features đã lọc

In [ ]:
assert len(kept_idx) >= 1, 'Không có feature nào qua filter! Hạ CORR_THRESHOLD xuống.'

scaler    = StandardScaler()
X_filt_s  = scaler.fit_transform(X_filt)

comp_results = []
for nc in range(1, min(MAX_COMPONENTS, X_filt.shape[1], len(y)-1) + 1):
    pls  = PLSRegression(n_components=nc)
    yp   = cross_val_predict(pls, X_filt_s, y, cv=LeaveOneOut()).ravel()
    r2   = r2_score(y, yp)
    rmse = np.sqrt(mean_squared_error(y, yp))
    comp_results.append({'n_comp': nc, 'R2_LOO': round(r2,4), 'RMSE_LOO': round(rmse,4)})

comp_df  = pd.DataFrame(comp_results)
best_nc  = comp_df.loc[comp_df['R2_LOO'].idxmax(), 'n_comp']
print(f'Features vào PLS: {len(kept_idx)}')
print(comp_df.to_string(index=False))
print(f'\nAuto-selected n_components = {best_nc}')

pls_step3   = PLSRegression(n_components=best_nc)
pls_step3.fit(X_filt_s, y)
y_loo_step3 = cross_val_predict(pls_step3, X_filt_s, y, cv=LeaveOneOut()).ravel()
r2_step3    = r2_score(y, y_loo_step3)
rmse_step3  = np.sqrt(mean_squared_error(y, y_loo_step3))
print(f'\nPLS (corr-filtered): R²={r2_step3:.4f}  RMSE={rmse_step3:.4f} mM  ({len(kept_idx)} features)')

---
## 4. VIP Scores

In [ ]:
def calc_vip(model):
    t, w, q = model.x_scores_, model.x_weights_, model.y_loadings_
    p, h    = w.shape
    s       = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    return np.array([
        np.sqrt(p * (s.T @ np.array([(w[i,j]/np.linalg.norm(w[:,j]))**2
                for j in range(h)])) / np.sum(s))[0]
        for i in range(p)
    ])

vip     = calc_vip(pls_step3)
vip_df  = pd.DataFrame({
    'feature'  : kept_names,
    'VIP'      : np.round(vip, 4),
    '|r|'      : [round(corr_df.loc[corr_df['feature']==n, 'abs_r'].values[0], 4)
                  for n in kept_names]
}).sort_values('VIP', ascending=False).reset_index(drop=True)

print('=== VIP Scores ===')
print(vip_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(max(8, len(vip_df)*0.5), 5))
ax.bar(range(len(vip_df)), vip_df['VIP'],
       color=['#e74c3c' if i < VIP_TOP_N else '#95a5a6' for i in range(len(vip_df))],
       alpha=0.85, edgecolor='black', lw=0.5)
ax.axhline(1.0, color='gray', lw=1.2, linestyle='--', label='VIP=1.0')
ax.axvline(VIP_TOP_N - 0.5, color='red', lw=2, linestyle='--', label=f'Top {VIP_TOP_N} cutoff')
ax.set_xticks(range(len(vip_df)))
ax.set_xticklabels(vip_df['feature'], rotation=70, fontsize=8)
ax.set_ylabel('VIP Score'); ax.set_title('VIP Scores (red = selected Top N)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('pipeline_vip.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Final PLS với Top N VIP features

In [ ]:
n_select       = min(VIP_TOP_N, len(vip_df))
selected_names = vip_df.head(n_select)['feature'].tolist()
selected_idx2  = [kept_names.index(f) for f in selected_names]
X_sel          = X_filt[:, selected_idx2]
X_sel_s        = StandardScaler().fit_transform(X_sel)

# Chọn n_components tốt nhất
comp_final = []
for nc in range(1, min(MAX_COMPONENTS, X_sel.shape[1], len(y)-1) + 1):
    pls  = PLSRegression(n_components=nc)
    yp   = cross_val_predict(pls, X_sel_s, y, cv=LeaveOneOut()).ravel()
    r2   = r2_score(y, yp)
    rmse = np.sqrt(mean_squared_error(y, yp))
    comp_final.append({'n_comp': nc, 'R2_LOO': round(r2,4), 'RMSE_LOO': round(rmse,4)})

comp_final_df = pd.DataFrame(comp_final)
best_nc_final = comp_final_df.loc[comp_final_df['R2_LOO'].idxmax(), 'n_comp']
print(comp_final_df.to_string(index=False))

# Train
pls_final  = PLSRegression(n_components=best_nc_final)
pls_final.fit(X_sel_s, y)
y_train    = pls_final.predict(X_sel_s).ravel()
y_loo      = cross_val_predict(pls_final, X_sel_s, y, cv=LeaveOneOut()).ravel()
r2_train   = r2_score(y, y_train)
r2_loo     = r2_score(y, y_loo)
rmse_train = np.sqrt(mean_squared_error(y, y_train))
rmse_loo   = np.sqrt(mean_squared_error(y, y_loo))

print('\n' + '='*52)
print(f'  FINAL PLS  (nComp={best_nc_final}, nFeat={n_select})')
print(f'  Selected   : {selected_names}')
print(f'  Train  R²  = {r2_train:.4f}   RMSE = {rmse_train:.4f} mM')
print(f'  LOO-CV R²  = {r2_loo:.4f}   RMSE = {rmse_loo:.4f} mM')
print('='*52)
print(f'\n  Before VIP selection : R²={r2_step3:.4f}  ({len(kept_idx)} features)')
print(f'  After  VIP selection : R²={r2_loo:.4f}  ({n_select} features)')
print(f'  ΔR² = {r2_loo - r2_step3:+.4f}')

---
## 6. Visualization

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, hspace=0.42, wspace=0.35)

# 6.1 Predicted vs Actual
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(y, y_loo, s=100, color='steelblue', zorder=5)
lim = [y.min()-0.4, y.max()+0.4]
ax1.plot(lim, lim, 'r--', lw=1.5, label='Ideal')
for a, p in zip(y, y_loo):
    ax1.annotate(f'{a}', (a, p), textcoords='offset points', xytext=(5,4), fontsize=8)
ax1.set_xlabel('Actual'); ax1.set_ylabel('Predicted')
ax1.set_title(f'Predicted vs Actual (LOO)\nR²={r2_loo:.4f}  RMSE={rmse_loo:.4f}', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

# 6.2 Residuals
ax2 = fig.add_subplot(gs[0, 1])
res = y_loo - y
ax2.bar(range(len(y)), res,
        color=['#e74c3c' if r < 0 else '#3498db' for r in res],
        alpha=0.85, edgecolor='black')
ax2.axhline(0, color='black', lw=1)
ax2.set_xticks(range(len(y)))
ax2.set_xticklabels([str(v) for v in y], fontsize=9)
for i, ri in enumerate(res):
    ax2.annotate(f'{ri:.3f}', (i, ri), textcoords='offset points',
                 xytext=(0, 5 if ri >= 0 else -13), ha='center', fontsize=8)
ax2.set_xlabel('Sample'); ax2.set_ylabel('Residual')
ax2.set_title('Residuals (LOO-CV)', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# 6.3 VIP final
ax3 = fig.add_subplot(gs[0, 2])
vip_final  = calc_vip(pls_final)
short_labs = selected_names
ax3.bar(range(len(vip_final)), vip_final,
        color=['#e74c3c' if v >= 1.0 else '#95a5a6' for v in vip_final],
        alpha=0.85, edgecolor='black')
ax3.axhline(1.0, color='red', lw=1.5, linestyle='--', label='VIP=1.0')
ax3.set_xticks(range(len(selected_names)))
ax3.set_xticklabels(short_labs, rotation=60, fontsize=8)
ax3.set_ylabel('VIP Score'); ax3.set_title('VIP (Final Model)', fontweight='bold')
ax3.legend(); ax3.grid(True, alpha=0.3, axis='y')

# 6.4 PLS Loadings
ax4 = fig.add_subplot(gs[1, 0])
for nc_i in range(best_nc_final):
    ax4.plot(range(len(selected_names)), pls_final.x_rotations_[:, nc_i],
             'o-', lw=1.8, ms=6, label=f'Comp {nc_i+1}')
ax4.axhline(0, color='black', lw=0.8)
ax4.set_xticks(range(len(selected_names)))
ax4.set_xticklabels(short_labs, rotation=60, fontsize=8)
ax4.set_ylabel('Loading'); ax4.set_title('PLS Loadings (W*)', fontweight='bold')
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

# 6.5 PLS Scores
ax5 = fig.add_subplot(gs[1, 1])
T = pls_final.x_scores_
if best_nc_final >= 2:
    sc = ax5.scatter(T[:,0], T[:,1], c=y, cmap='viridis', s=120, edgecolor='black', zorder=5)
    plt.colorbar(sc, ax=ax5, label='y')
    for i, c in enumerate(y):
        ax5.annotate(str(c), (T[i,0], T[i,1]), textcoords='offset points', xytext=(5,3), fontsize=8)
    ax5.set_xlabel('Comp 1'); ax5.set_ylabel('Comp 2')
    ax5.set_title('PLS Scores (T1 vs T2)', fontweight='bold')
else:
    ax5.scatter(y, T[:,0], c=y, cmap='viridis', s=120, edgecolor='black', zorder=5)
    for i, c in enumerate(y):
        ax5.annotate(str(c), (c, T[i,0]), textcoords='offset points', xytext=(5,3), fontsize=8)
    ax5.set_xlabel('y (actual)'); ax5.set_ylabel('Comp 1')
    ax5.set_title('PLS Score T1 vs y', fontweight='bold')
ax5.grid(True, alpha=0.3)

# 6.6 Pipeline Summary
ax6 = fig.add_subplot(gs[1, 2])
stages    = [f'All\n({len(feature_cols)})', f'Corr\n({len(kept_idx)})', f'VIP Top{n_select}\n({n_select})']
r2_stages = [np.nan, r2_step3, r2_loo]
colors3   = ['#bdc3c7', '#3498db', '#e74c3c']
bars = ax6.bar([0,1,2], [0 if np.isnan(v) else v for v in r2_stages],
               color=colors3, alpha=0.75, edgecolor='black', width=0.5)
for i, v in enumerate(r2_stages):
    if not np.isnan(v):
        ax6.text(i, v+0.02, f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')
ax6.set_xticks([0,1,2]); ax6.set_xticklabels(stages, fontsize=9)
ax6.set_ylabel('R² (LOO-CV)'); ax6.set_ylim(0, 1.15)
ax6.set_title('Pipeline Summary', fontweight='bold')
ax6.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    f'PLS Pipeline  |  {n_select} features  nComp={best_nc_final}  '
    f'R²={r2_loo:.4f}  RMSE={rmse_loo:.4f}',
    fontsize=11, fontweight='bold'
)
plt.savefig('pipeline_final.png', dpi=150, bbox_inches='tight')
plt.show()

# Result table
print('\n=== Prediction Table ===')
df_res = pd.DataFrame({
    'Actual'     : y,
    'Pred_LOO'   : y_loo.round(4),
    'Pred_Train' : y_train.round(4),
    'Residual'   : (y_loo-y).round(4),
    'AbsError'   : np.abs(y_loo-y).round(4)
})
print(df_res.to_string(index=False))
print(f'\nMean Abs Error : {np.abs(y_loo-y).mean():.4f}')